In [1]:
######################################################################################################################
# Notebook: 02_Construccion_Corpus_Arancel2022_RGI_Notas.ipynb
# Autor del código: Vladimir Molleapasa Gutierrez
# Fecha de generación: 13/01/2026
# Código generado con asistencia de ChatGPT 5.2 Thinking
# Prompt original: "Adjunto Arancel 2022.pdf. Genera un notebook 02_Construccion_Corpus_Arancel2022_RGI_Notas 
#                   que extraiga el bloque RGI y lo separe en reglas 1..6 (más contexto opcional), 
#                   extraiga Notas de Sección y Capítulo (sin tabla arancelaria), exporte arancel2022_rgi.jsonl y 
#                   arancel2022_notas.jsonl + run_metadata.json (SHA-256/entorno/parámetros) + summary.csv (QA). 
#                   Doc_id único y documentación para reproducibilidad"
# Autor del prompt: Vladimir Molleapasa Gutierrez
# Ajustes y validación: Vladimir Molleapasa Gutierrez
# Uso académico, con revisión propia del autor.
# Licencia: Uso académico, no comercial.
######################################################################################################################

# =============================================================================
# Propósito:
#   Este notebook construye parte del corpus documental para un enfoque RAG
#   (Retrieval-Augmented Generation) orientado a clasificación arancelaria.
#   A partir del PDF oficial "Arancel 2022.pdf" se extraen dos componentes
#   normativos fundamentales:
#
#     (1) RGI: Reglas Generales para la Interpretación de la Nomenclatura
#     (2) Notas legales: Notas de Sección y Notas de Capítulo
#
#   Estos textos se transforman a formato JSONL para ser indexados por un motor
#   de recuperación (BM25 y/o embeddings) y usados como evidencia en RAG.
#
# Entradas:
#   - data/raw/Arancel 2022.pdf
#
# Salidas (artefactos reproducibles):
#   - data/processed/arancel2022_rgi.jsonl
#       Un documento por regla (Regla 1..6), con campos: doc_id, tipo, texto,
#       páginas de origen, fuente.
#   - data/processed/arancel2022_notas.jsonl
#       Un documento por nota (Sección o Capítulo), con campos: doc_id, tipo,
#       scope, sección/capítulo, texto, páginas de origen, fuente.
#   - data/processed/arancel2022_run_metadata.json
#       Metadatos del run para reproducibilidad:
#         * hash SHA-256 del PDF de entrada
#         * fecha/hora UTC de ejecución
#         * versión de Python y librerías
#         * parámetros del parser (regex, límites de páginas, rutas)
#   - data/processed/arancel2022_summary.csv
#       QA rápido: conteos de documentos extraídos por tipo y métricas mínimas.
#
# Reproducibilidad:
#   - El hash SHA-256 del PDF permite demostrar que el corpus fue generado
#     exactamente desde un insumo específico.
#   - Los parámetros de extracción (regex de inicio/fin RGI, regex de notas,
#     corte por aparición de códigos arancelarios) se registran en metadata.
#
# Dependencias:
#   - pypdf: extracción de texto desde PDFs basados en texto (no escaneados).
#   - tqdm: barra de progreso (útil para ejecución controlada y trazable).
#
# Instalación (si aplica en el entorno local):
#   !pip install pypdf tqdm
#
# Limitaciones conocidas:
#   - Si el PDF contiene páginas escaneadas (sin texto seleccionable), pypdf
#     puede retornar texto vacío; en ese caso se requiere OCR (no incluido en
#     este notebook, dado que el piloto se orienta a PDFs "text-based").
# =============================================================================

# Imports principales (se mantienen en esta celda para control explícito del entorno)
from pathlib import Path
import re
import json
import hashlib
import platform
from datetime import datetime

from pypdf import PdfReader
from tqdm import tqdm


In [2]:
# =============================================================================
# Configuración (rutas del repositorio)
# =============================================================================

# Raíz del proyecto/repo (ajustar si cambia la ubicación local)
PROJECT_ROOT = Path(r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_RAW / "Arancel 2022.pdf"

# Salidas (módulos separados; luego se pueden unificar en un corpus final si se desea)
OUT_RGI_JSONL = DATA_PROCESSED / "arancel2022_rgi.jsonl"
OUT_NOTAS_JSONL = DATA_PROCESSED / "arancel2022_notas.jsonl"
OUT_RUN_METADATA = DATA_PROCESSED / "arancel2022_run_metadata.json"
OUT_SUMMARY_CSV = DATA_PROCESSED / "arancel2022_summary.csv"

# Control de ejecución (útil para pruebas rápidas)
MAX_PAGES = None  # None = todas las páginas; int = primeras N páginas

assert PDF_PATH.exists(), f"No se encuentra el PDF en: {PDF_PATH}"

print("Input PDF:", PDF_PATH)
print("Output RGI:", OUT_RGI_JSONL)
print("Output Notas:", OUT_NOTAS_JSONL)
print("Output Run Metadata:", OUT_RUN_METADATA)
print("Output Summary:", OUT_SUMMARY_CSV)


Input PDF: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\raw\Arancel 2022.pdf
Output RGI: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_rgi.jsonl
Output Notas: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_notas.jsonl
Output Run Metadata: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_run_metadata.json
Output Summary: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_summary.csv


In [3]:
# =============================================================================
# Utilidades de reproducibilidad
# =============================================================================
import hashlib
import json
import platform
from datetime import datetime
from typing import Dict, Any, Iterable

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calcula SHA-256 del archivo (trazabilidad exacta del insumo)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def environment_metadata() -> Dict[str, Any]:
    """Captura metadatos del entorno para reproducibilidad."""
    try:
        from importlib.metadata import version as pkg_version
        pypdf_ver = pkg_version("pypdf")
        tqdm_ver = pkg_version("tqdm")
    except Exception:
        pypdf_ver = None
        tqdm_ver = None

    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "libraries": {
            "pypdf": pypdf_ver,
            "tqdm": tqdm_ver,
        }
    }

def write_jsonl(records: Iterable[Dict[str, Any]], out_path: Path) -> None:
    """Escribe un archivo JSONL (una línea JSON por registro)."""
    with out_path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [4]:
# =============================================================================
# Lectura del PDF y extracción de texto por página
# =============================================================================
from pypdf import PdfReader
from tqdm import tqdm

def iter_pages_text(pdf_path: Path, max_pages=None):
    """
    Itera el texto de cada página del PDF.

    Retorna tuplas:
      (page_no_1based, page_text)

    Notas:
      - Si una página no tiene texto extraíble, se retorna string vacío.
      - Para el parser de notas, la granularidad por línea se obtiene con splitlines().
    """
    reader = PdfReader(str(pdf_path))
    n_pages_total = len(reader.pages)
    last = n_pages_total if max_pages is None else min(max_pages, n_pages_total)

    for i in tqdm(range(last), desc="Extrayendo texto del PDF"):
        page_no = i + 1
        text = reader.pages[i].extract_text() or ""
        # Normalización ligera (sin destruir estructura)
        text = text.replace("\u00a0", " ").strip()
        yield page_no, text

# Validación rápida de número de páginas (sin recorrer todo el texto)
reader_tmp = PdfReader(str(PDF_PATH))
print("Total páginas:", len(reader_tmp.pages))


Total páginas: 434


In [5]:
# =============================================================================
# Extracción de RGI (bloque) y segmentación por regla
# =============================================================================
import re
from typing import List, Tuple

RGI_START_RE = re.compile(r"REGLAS\s+GENERALES\s+PARA\s+LA\s+INTERPRETACI[ÓO]N\s+DE\s+LA\s+NOMENCLATURA", re.IGNORECASE)
RGI_END_RE   = re.compile(r"REGLAS\s+PARA\s+LA\s+APLICACI[ÓO]N\s+DEL\s+ARANCEL", re.IGNORECASE)

RULE_START_RE = re.compile(r"(?m)^\s*([1-6])\.\s+")

def extract_rgi_block(pages_text: Iterable[Tuple[int, str]]) -> Tuple[str, int, int]:
    """
    Localiza y extrae el bloque completo de RGI a partir de marcadores de inicio/fin.

    Retorna:
      (rgi_text, page_start, page_end)

    Comportamiento:
      - Se inicia cuando se encuentra el marcador de inicio.
      - Finaliza al encontrar el marcador de fin; el texto posterior al marcador no se incluye.
    """
    collecting = False
    buf: List[str] = []
    page_start = None
    page_end = None

    for page_no, text in pages_text:
        if not text:
            continue

        if not collecting:
            m = RGI_START_RE.search(text)
            if m:
                collecting = True
                page_start = page_no
                # Tomar desde el inicio del bloque en esta página
                buf.append(text[m.start():])
                page_end = page_no
        else:
            # Si ya se está recolectando, buscar fin
            m_end = RGI_END_RE.search(text)
            if m_end:
                buf.append(text[:m_end.start()])
                page_end = page_no
                break
            buf.append(text)
            page_end = page_no

    if not collecting or page_start is None or page_end is None:
        raise RuntimeError("No se pudo localizar el bloque RGI (marcadores no encontrados).")

    rgi_text = "\n".join(buf)
    return rgi_text, page_start, page_end


def split_rgi_rules(rgi_text: str, page_start: int, page_end: int, source_name: str) -> List[dict]:
    """
    Segmenta el bloque RGI en reglas 1..6.

    Salida: lista de dicts listos para JSONL (un documento por regla).
    """
    # Normalización moderada (preserva numeración y saltos)
    text = re.sub(r"\r\n", "\n", rgi_text)
    text = re.sub(r"[ \t]+", " ", text)

    matches = list(RULE_START_RE.finditer(text))
    if not matches:
        raise RuntimeError("No se detectaron reglas numeradas 1..6 en el bloque RGI.")

    records = []
    for idx, m in enumerate(matches):
        rule_no = m.group(1)
        start = m.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)

        rule_text = text[start:end].strip()

        records.append({
            "doc_id": f"ARANCEL2022_RGI_{rule_no}",
            "tipo": "rgi",
            "ref": f"RGI {rule_no}",
            "titulo": f"Regla General {rule_no}",
            "texto": rule_text,
            "pagina_inicio": page_start,
            "pagina_fin": page_end,
            "fuente": source_name,
            "idioma": "es",
            "version": "Arancel_2022",
        })

    return records


In [6]:
# =============================================================================
# Extracción de Notas: Sección y Capítulo
# =============================================================================
from typing import Optional

SECTION_RE = re.compile(r"^\s*Secci[óo]n\s+([IVXLC]+)\b", re.IGNORECASE)
CHAPTER_RE = re.compile(r"^\s*Cap[ií]tulo\s+(\d{1,2})\b", re.IGNORECASE)

# Encabezado típico de notas en el Arancel: "Nota." o "Notas."
NOTE_HEADER_RE = re.compile(r"^\s*Notas?\.\s*$", re.IGNORECASE)

# Primera línea típica de códigos arancelarios (10 dígitos con puntos): 1108.11.00.00
TARIFF_CODE_RE = re.compile(r"^\s*\d{4}(?:\.\d{2}){3}\b")

def normalize_line(line: str) -> str:
    """Normalización ligera de línea (evita colapsar estructura)."""
    line = line.replace("\u00a0", " ")
    line = re.sub(r"[ \t]+", " ", line).strip()
    return line

def extract_notas_records(pages_text: Iterable[Tuple[int, str]], source_name: str) -> List[dict]:
    """
    Extrae:
      - Nota(s) inmediatamente asociadas a cada Sección
      - Nota(s) inmediatamente asociadas a cada Capítulo

    Estrategia (basada en la estructura típica del Arancel):
      - Cuando se detecta "Sección <ROMANO>", se fija contexto de sección y se acumula
        el título de sección (línea siguiente no vacía).
      - Se inicia captura de nota de sección al encontrar "Nota." o "Notas." (encabezado).
      - La nota de sección termina cuando aparece "Capítulo <n>" (inicio del primer capítulo).
      - Para capítulos: se detecta "Capítulo <n>", se registra título y se inicia captura al
        encontrar "Nota." o "Notas.".
      - La nota de capítulo termina al encontrar la primera línea con un código arancelario
        (inicio de partidas/subpartidas), o el siguiente capítulo/sección.
    """
    records: List[dict] = []

    current_section: Optional[str] = None
    current_section_title: Optional[str] = None

    current_chapter: Optional[str] = None
    current_chapter_title: Optional[str] = None

    capturing: Optional[str] = None  # None | "nota_seccion" | "nota_capitulo"
    buffer: List[str] = []
    page_start: Optional[int] = None
    page_end: Optional[int] = None

    # Flags para capturar títulos tras encabezados
    expect_section_title = False
    expect_chapter_title = False

    def finalize_note():
        """Cierra el bloque en captura y crea un registro JSON serializable."""
        nonlocal capturing, buffer, page_start, page_end

        if capturing is None or page_start is None or page_end is None:
            capturing = None
            buffer = []
            page_start = None
            page_end = None
            return

        text = "\n".join(buffer).strip()
        if not text:
            capturing = None
            buffer = []
            page_start = None
            page_end = None
            return

        if capturing == "nota_seccion":
            doc_id = f"ARANCEL2022_NOTA_SECCION_{current_section or 'NA'}"
            titulo = f"Notas de Sección {current_section or ''}".strip()
            ref = f"Sección {current_section}" if current_section else "Sección NA"
            scope = "seccion"
        else:
            doc_id = f"ARANCEL2022_NOTA_CAPITULO_{current_chapter or 'NA'}"
            titulo = f"Notas de Capítulo {current_chapter or ''}".strip()
            ref = f"Capítulo {current_chapter}" if current_chapter else "Capítulo NA"
            scope = "capitulo"

        records.append({
            "doc_id": doc_id,
            "tipo": capturing,
            "scope": scope,
            "ref": ref,
            "titulo": titulo,
            "section": current_section,
            "section_title": current_section_title,
            "chapter": current_chapter,
            "chapter_title": current_chapter_title,
            "texto": text,
            "pagina_inicio": page_start,
            "pagina_fin": page_end,
            "fuente": source_name,
            "idioma": "es",
            "version": "Arancel_2022",
        })

        capturing = None
        buffer = []
        page_start = None
        page_end = None

    for page_no, text in pages_text:
        if not text:
            continue

        lines = [normalize_line(ln) for ln in text.splitlines() if normalize_line(ln)]

        for line in lines:
            # Detectar SECCIÓN
            msec = SECTION_RE.match(line)
            if msec:
                # cierre preventivo de notas si existiese captura abierta
                finalize_note()

                current_section = msec.group(1).upper()
                current_section_title = None

                # reinicio de contexto de capítulo al cambiar de sección
                current_chapter = None
                current_chapter_title = None

                expect_section_title = True
                expect_chapter_title = False
                continue

            # Captura del título de sección (primera línea tras el encabezado "Sección X")
            if expect_section_title:
                # Si la línea ya es el encabezado de notas, se asume título omitido
                if NOTE_HEADER_RE.match(line):
                    expect_section_title = False
                    capturing = "nota_seccion"
                    buffer = [line]  # conserva encabezado "Nota(s)."
                    page_start = page_no
                    page_end = page_no
                else:
                    current_section_title = line
                    expect_section_title = False
                continue

            # Detectar CAPÍTULO
            mcap = CHAPTER_RE.match(line)
            if mcap:
                # si se estaba capturando nota de sección, termina al iniciar capítulo
                finalize_note()

                current_chapter = mcap.group(1).zfill(2)
                current_chapter_title = None

                expect_chapter_title = True
                continue

            # Captura del título de capítulo (primera línea tras "Capítulo n")
            if expect_chapter_title:
                if NOTE_HEADER_RE.match(line):
                    expect_chapter_title = False
                    capturing = "nota_capitulo"
                    buffer = [line]
                    page_start = page_no
                    page_end = page_no
                else:
                    current_chapter_title = line
                    expect_chapter_title = False
                continue

            # Inicio explícito de notas si aparece "Nota(s)." en contexto ya definido
            if capturing is None and NOTE_HEADER_RE.match(line):
                # Heurística:
                # - Si hay capítulo actual (y título ya leído), se considera nota de capítulo.
                # - Si no hay capítulo actual pero sí sección, se considera nota de sección.
                if current_chapter is not None:
                    capturing = "nota_capitulo"
                elif current_section is not None:
                    capturing = "nota_seccion"
                else:
                    # Caso fuera de contexto: se ignora para evitar ruido del índice.
                    continue

                buffer = [line]
                page_start = page_no
                page_end = page_no
                continue

            # Cierre de nota de capítulo al iniciar códigos arancelarios
            if capturing == "nota_capitulo" and TARIFF_CODE_RE.match(line):
                finalize_note()
                # No se consume el código; el corpus de notas no necesita la tabla arancelaria.
                continue

            # Acumulación de contenido durante captura
            if capturing in {"nota_seccion", "nota_capitulo"}:
                buffer.append(line)
                page_end = page_no
                continue

            # Fuera de captura: se ignora el resto
            continue

    # cierre final si quedó bloque abierto
    finalize_note()
    return records


In [7]:
# =============================================================================
# Ejecución: Arancel 2022 -> JSONL (RGI + Notas) + Metadata + QA Summary
# =============================================================================
import time
from collections import Counter

start_time = time.time()

pdf_sha256 = sha256_file(PDF_PATH)
source_name = PDF_PATH.name

# 1) RGI (solo requiere recorrer pocas páginas; se obtiene desde el stream completo)
pages_for_rgi = list(iter_pages_text(PDF_PATH, max_pages=MAX_PAGES))
rgi_text, rgi_page_start, rgi_page_end = extract_rgi_block(iter(pages_for_rgi))
rgi_records = split_rgi_rules(rgi_text, rgi_page_start, rgi_page_end, source_name)

# 2) Notas legales (recorre todas las páginas para identificar secciones/capítulos)
#    Se usa un iterador nuevo para asegurar recorrido completo y consistente.
pages_for_notas = iter_pages_text(PDF_PATH, max_pages=MAX_PAGES)
notas_records = extract_notas_records(pages_for_notas, source_name)

# 3) Escritura JSONL
write_jsonl(rgi_records, OUT_RGI_JSONL)
write_jsonl(notas_records, OUT_NOTAS_JSONL)

# 4) QA summary (conteos y checks mínimos)
counts = Counter([rec["tipo"] for rec in (rgi_records + notas_records)])
unique_sections = sorted({rec.get("section") for rec in notas_records if rec.get("section")})
unique_chapters = sorted({rec.get("chapter") for rec in notas_records if rec.get("chapter")})

qa = {
    "input_pdf": source_name,
    "sha256": pdf_sha256,
    "records_total": len(rgi_records) + len(notas_records),
    "counts_by_tipo": dict(counts),
    "rgi_rules_extracted": len(rgi_records),
    "unique_sections_in_notas": [s for s in unique_sections if s],
    "unique_chapters_in_notas": [c for c in unique_chapters if c],
    "notas_short_records": [
        rec["doc_id"] for rec in notas_records
        if len(rec.get("texto","")) < 200
    ][:50],  # se limita para no inflar
}

# Summary CSV simple (para revisión rápida en Excel)
lines = ["metric,value"]
lines.append(f"records_total,{qa['records_total']}")
lines.append(f"rgi_rules_extracted,{qa['rgi_rules_extracted']}")
lines.append(f"notas_records,{len(notas_records)}")
for k, v in qa["counts_by_tipo"].items():
    lines.append(f"count_{k},{v}")
lines.append(f"unique_sections_in_notas,{len(qa['unique_sections_in_notas'])}")
lines.append(f"unique_chapters_in_notas,{len(qa['unique_chapters_in_notas'])}")
lines.append(f"notas_short_records_sample,{len(qa['notas_short_records'])}")

OUT_SUMMARY_CSV.write_text("\n".join(lines), encoding="utf-8")

# 5) Run metadata (reproducibilidad)
run_meta = {
    "input": {
        "path": str(PDF_PATH),
        "name": source_name,
        "sha256": pdf_sha256,
    },
    "environment": environment_metadata(),
    "parameters": {
        "max_pages": MAX_PAGES,
        "outputs": {
            "rgi_jsonl": str(OUT_RGI_JSONL),
            "notas_jsonl": str(OUT_NOTAS_JSONL),
            "run_metadata": str(OUT_RUN_METADATA),
            "summary_csv": str(OUT_SUMMARY_CSV),
        },
        "parser": {
            "engine": "pypdf",
            "rgi_markers": {
                "start_regex": RGI_START_RE.pattern,
                "end_regex": RGI_END_RE.pattern,
                "rule_split_regex": RULE_START_RE.pattern,
            },
            "notas_markers": {
                "section_regex": SECTION_RE.pattern,
                "chapter_regex": CHAPTER_RE.pattern,
                "note_header_regex": NOTE_HEADER_RE.pattern,
                "tariff_code_regex": TARIFF_CODE_RE.pattern,
            },
        },
    },
    "qa": qa,
    "runtime_seconds": round(time.time() - start_time, 3),
}

OUT_RUN_METADATA.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

print("=== Extracción completada ===")
print("RGI reglas:", len(rgi_records), "->", OUT_RGI_JSONL)
print("Notas registros:", len(notas_records), "->", OUT_NOTAS_JSONL)
print("Run metadata:", OUT_RUN_METADATA)
print("Summary:", OUT_SUMMARY_CSV)
print("SHA-256 PDF:", pdf_sha256)


Extrayendo texto del PDF: 100%|██████████| 434/434 [00:26<00:00, 16.30it/s]

=== Extracción completada ===
RGI reglas: 8 -> C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_rgi.jsonl
Notas registros: 96 -> C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_notas.jsonl
Run metadata: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_run_metadata.json
Summary: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_summary.csv
SHA-256 PDF: a01a029e1ca29b6debc61d219c17dfc086354e00669246cc24a91ad9f454c7d0


In [8]:
# =============================================================================
# Inspección rápida (muestras)
# =============================================================================
import itertools

def head_jsonl(path: Path, n=3):
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in itertools.islice(f, n):
            out.append(json.loads(line))
    return out

print("Muestra RGI:")
for rec in head_jsonl(OUT_RGI_JSONL, 2):
    print(rec["doc_id"], "|", rec["titulo"])
    print(rec["texto"][:300], "...\n")

print("Muestra Notas:")
for rec in head_jsonl(OUT_NOTAS_JSONL, 2):
    print(rec["doc_id"], "|", rec["titulo"], "|", rec.get("ref"))
    print(rec["texto"][:300], "...\n")


Muestra RGI:
ARANCEL2022_RGI_1 | Regla General 1
1. El 1° de enero de 1988 entró en vigencia el Convenio Internacional sobre el Sistema 
Armonizado de Designación y Codificación de Mercancías (Convenio) del Consejo de 
Cooperación Aduanera, actualmente Organización Mundial de Aduanas (OMA), cuyo 
Anexo comprende la Nomenclatura del Sistema Armoniz ...

ARANCEL2022_RGI_2 | Regla General 2
2. La NANDINA constituye la Nomenclatura Arancelaria Común de la Comunidad Andina 
y está basada en la VUESA. 
 
El código numérico de la NANDINA está compuesto de ocho dígitos. Si una subpartida 
del Sistema Armonizado no se ha subdividido por necesidades comunitarias, los dígitos 
séptimo (7°) y o ...

Muestra Notas:
ARANCEL2022_NOTA_SECCION_I | Notas de Sección I | Sección I
Notas.
1. En esta Sección, cualquier referencia a un género o a una especie determinada de un animal se aplica
también, salvo disposición en contrario, a los animales jóvenes de ese género o de esa especie.
2. Salvo disposición 